# Assignment 3 — Work File
## Two Averages, One Argument

This notebook contains the descriptive-statistics work, an interpretation of the results, and a classmate challenge.

**Dataset:** `teaching_prices_clean.csv`

**Important data note:** The CSV labels itself as *illustrative teaching data — not actual PBS observations*. The results below describe only this teaching dataset and must not be presented as official market-price evidence.

**How to run:** Keep this notebook in the same folder as `teaching_prices_clean.csv`, then run the cells from top to bottom. If the CSV is elsewhere, change the file path in the next cell.

In [ ]:
import pandas as pd
from pathlib import Path

# Keep teaching_prices_clean.csv in the same folder as this notebook.
csv_path = Path("teaching_prices_clean.csv")

if not csv_path.exists():
    raise FileNotFoundError(
        "Could not find teaching_prices_clean.csv. "
        "Place it in the same folder as this notebook or update csv_path."
    )

df = pd.read_csv(csv_path)
df["date"] = pd.to_datetime(df["date"])

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
display(df.head())


## 1. Check the dataset

The file contains four items, two cities, and twelve weekly dates. Each row records one item-city price observation.

In [ ]:
print("Number of observations:", len(df))
print("Items:", sorted(df["item"].unique()))
print("Cities:", sorted(df["city"].unique()))
print("Number of dates:", df["date"].nunique())
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("\nUnits:", df["unit"].unique())
print("\nData label(s):")
display(df["data_label"].value_counts())


## 2. Mean and median for each item

The mean is the arithmetic average of all recorded prices for an item. The median is the middle price after sorting the observations. Here, each item has 24 observations (12 dates × 2 cities).

In [ ]:
summary = (
    df.groupby("item")["price"]
      .agg(observations="count", mean="mean", median="median")
      .reset_index()
)

summary["mean"] = summary["mean"].round(2)
summary["median"] = summary["median"].round(2)
display(summary)


## 3. First-to-last percentage change

For each item, first calculate the average price across the two cities on the first date and on the last date. Then calculate:

\[
\text{Percentage change} =
\frac{\text{Last-date average} - \text{First-date average}}
{\text{First-date average}} \times 100
\]

In [ ]:
date_city_avg = (
    df.groupby(["item", "date"], as_index=False)["price"]
      .mean()
)

first_last = (
    date_city_avg.sort_values("date")
      .groupby("item")
      .agg(
          first_date=("date", "first"),
          first_date_average=("price", "first"),
          last_date=("date", "last"),
          last_date_average=("price", "last")
      )
      .reset_index()
)

first_last["percentage_change"] = (
    (first_last["last_date_average"] - first_last["first_date_average"])
    / first_last["first_date_average"] * 100
)

first_last[["first_date_average", "last_date_average", "percentage_change"]] = (
    first_last[["first_date_average", "last_date_average", "percentage_change"]].round(2)
)

display(first_last)


## 4. Combined results table

In [ ]:
results = summary.merge(first_last, on="item")
results = results[[
    "item", "observations", "mean", "median",
    "first_date_average", "last_date_average", "percentage_change"
]]
display(results)


## 5. Interpretation of the results

- All four items have a higher average price at the final date than at the first date.
- Sweetener D has the largest first-to-last percentage increase.
- Pulse C has the smallest first-to-last percentage increase.
- Oil B has the highest overall mean price level. A higher price level is not the same as the fastest percentage increase.
- The mean and median are close for all four items, so they give similar central-value summaries in this dataset.

### Possible explanations (not established by the data)

Supply changes, demand changes, seasonality, transport costs, or market conditions could contribute to price movements. The CSV does not contain variables that would allow these explanations to be tested. They must therefore be described as possibilities, not findings.

### Limitations

1. The dataset is illustrative and is not actual PBS data.
2. It contains only four teaching items and two teaching cities.
3. A first-to-last comparison does not describe every weekly movement.
4. The dataset contains no explanatory variables, so it cannot establish the causes of price changes.
5. The assignment's separate flagged-value comparison requires the Week 2 cleaned file or an identified flagged observation. That information is not present in this CSV.

## 6. Classmate challenge: “The mean is always the typical price.”

**Response:** Not always. The mean uses every observation and can be pulled upward or downward by unusually high or low values. The median is the middle value after sorting and is more resistant to extreme observations. In this dataset the mean and median are close, but that does not make the mean universally representative. A careful interpretation checks both measures and, where possible, also examines the spread and individual observations.

## 7. Final conclusion

The supplied teaching data show increases from the first to the last recorded date for all four items. Sweetener D has the largest percentage increase (+19.41%), while Pulse C has the smallest (+3.69%). These are descriptive findings about this illustrative dataset only. They do not prove why prices changed and should not be generalized to actual national market prices.